In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()


llm = ChatOpenAI(
    model="gpt-5",
    temperature=0.4,
    max_retries=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

llm.invoke("Hello, how are you?")

In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph
from pydantic import BaseModel, Field, ValidationError
from typing_extensions import Annotated, Literal


class State(BaseModel):
    route_selector: Annotated[
        Literal["a", "b", ""], Field(default="", description="The route selector")
    ]
    route_log: Annotated[
        list[str], Field(default_factory=list, description="The route log")
    ]
    visit_count: Annotated[int, Field(default=0, description="The visit count")]


def start_node(state: State) -> State:
    state.route_log.append("Start_node")
    return state


def route_select_to_a(state: State) -> State:
    state.route_log.append("Route_select_to_a")
    return state


def route_select_to_b(state: State) -> State:
    if "Route_select_to_b" in state.route_log:
        state.visit_count += 1
        state.route_log.append("Route_select_to_b is already in the route log")
    else:
        state.route_log.append("Route_select_to_b")

    return state


def route_selector(state: State) -> Literal["a", "b"]:
    if state.visit_count < 2:
        state.route_selector = "a"
        return state.route_selector
    else:
        state.route_selector = "b"
        return state.route_selector


def end_node(state: State) -> State:
    state.route_log.append("End_node")
    return state


config = RunnableConfig(
    recursion_limit=6,
    configurable={"thread_id": "1"},
    tags=["my-rag"],
)

memory = MemorySaver()

workflow = StateGraph(State)
workflow.add_node("start", start_node)
workflow.add_node("route_select_to_a", route_select_to_a)
workflow.add_node("route_select_to_b", route_select_to_b)
workflow.add_node("route_selector", route_selector)
workflow.add_node("end", end_node)

workflow.set_entry_point("start")
workflow.add_edge("start", "route_select_to_a")
workflow.add_edge("route_select_to_a", "route_select_to_b")
workflow.add_edge("route_select_to_b", "end")

graph = workflow.compile(checkpointer=memory)

try:
    graph.invoke(
        State(
            route_selector="",
            route_log=[],
            visit_count=0,
        ),
        config=config,
    )
except ValidationError as e:
    print(e.errors()[0])

graph.get_state(config=config)

In [ ]:
from pydantic import BaseModel, ValidationError


class Model(BaseModel):
    x: list[int]


try:
    Model(x=1)
except ValidationError as exc:
    print(exc.errors()[0])
    print(repr(exc.errors()[0]["type"]))
    # > 'list_type'

In [ ]:
from pydantic import BaseModel, Field
from typing_extensions import Annotated, Literal


class Test_state(BaseModel):
    x: Annotated[str, Field(description="x")]


def test_state(state: Test_state) -> Test_state:
    return state


config = RunnableConfig(
    configurable={"thread_id": "1"},
)

graph = StateGraph(Test_state)

NameError: name 'RunnableConfig' is not defined